# 獨立資料評估

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/07-heldout/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from miniyolo.data import ShapeDataset, collate
from miniyolo.models import GridDetector
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss
from miniyolo.inference import decode_grid
from miniyolo.metrics import evaluate_ap


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    targets = [{'boxes': torch.tensor([[8.,12.,24.,28.]]), 'labels': torch.tensor([0])},
               {'boxes': torch.tensor([[40.,36.,56.,52.]]), 'labels': torch.tensor([0])}]
    predictions = [
        {'boxes': torch.tensor([[8.,12.,24.,28.],[0.,0.,4.,4.],[8.,12.,24.,28.]]),
         'scores': torch.tensor([.9,.8,.7]), 'labels': torch.tensor([0,0,0])},
        {'boxes': torch.empty(0,4), 'scores': torch.empty(0),
         'labels': torch.empty(0,dtype=torch.long)}]
    metrics = evaluate_ap(predictions,targets,num_classes=2,iou_threshold=.5)
    assert abs(metrics['map']-.5) < 1e-6
    assert abs(metrics['precision']-1/3) < 1e-6
    assert abs(metrics['recall']-.5) < 1e-6
    assert metrics['ap_per_class'][1] is None
    print('artificial evaluation fixture', metrics)
    train = ShapeDataset(n=4,size=64,seed=7,max_objects=1,allow_empty=False)
    heldout = ShapeDataset(n=4,size=64,seed=901,max_objects=1,allow_empty=False)
    train_images, train_anns = collate([train[i] for i in range(4)])
    heldout_images, heldout_anns = collate([heldout[i] for i in range(4)])
    model = GridDetector(num_classes=2,grid_size=4,width=8)
    optimizer = torch.optim.Adam(model.parameters(),lr=.01)
    train_target = build_targets(train_anns,4,64,2)
    model.train()
    for _ in range(3):
        optimizer.zero_grad(set_to_none=True)
        loss = grid_loss(model(train_images),train_target)['total']
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        predicted = decode_grid(model(heldout_images), image_size=64, score_threshold=.01, nms_iou=.5)
    result = evaluate_ap(predicted,heldout_anns,num_classes=2,iou_threshold=.5)
    assert 0 <= result['map'] <= 1
    print('3-step held-out PIPELINE SMOKE, not trained detector evidence', result)


if __name__ == '__main__':
    main()


artificial evaluation fixture {'ap_per_class': {0: 0.5, 1: None}, 'map': 0.5, 'precision': 0.3333333333333333, 'recall': 0.5}
3-step held-out PIPELINE SMOKE, not trained detector evidence {'ap_per_class': {0: 0.0, 1: 0.0}, 'map': 0.0, 'precision': 0.0, 'recall': 0.0}
